In [43]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [44]:
    #reading data
df = pd.read_parquet('./Data/yellow_tripdata_2023-01.parquet')
df.head()


In [20]:
df.columns.tolist()
len(df.columns)


19

In [45]:
df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)


In [46]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,duration
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00,8.433333
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00,6.316667
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00,12.750000
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25,9.616667
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00,10.833333


In [47]:
#std deviation of duration
df['duration'].std()

np.float64(42.59435124195457)

In [52]:
#removing outlier
def remove_outlier_prepare(df):
    len(df)
    print(len(df))
    df_clean = df[(df.duration >= 1) & (df.duration <= 60)]
    print(len(df_clean))
    fraction = len(df_clean) / len(df)
    percentage = fraction * 100
    print("Fraction remaining:", fraction)
    print("Percentage remaining:", percentage, "%")
    categorical = ['PULocationID', 'DOLocationID']
    numerical = ['trip_distance']
    df_clean[categorical] = df_clean[categorical].astype(str)
    dicts = df_clean[categorical].to_dict(orient='records')
    dv = DictVectorizer()
    X = dv.fit_transform(dicts)
    print(X.shape)
    print("Number of columns:", X.shape[1])
    return df_clean


In [53]:
df_clean = remove_outlier_prepare(df)


3066766
3009173
Fraction remaining: 0.9812202822125979
Percentage remaining: 98.1220282212598 %
(3009173, 515)
Number of columns: 515


In [62]:
train_dicts = df_clean[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

target = 'duration'
y_train = df_clean[target].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)

root_mean_squared_error(y_train, y_pred)

7.6584039350980575

In [63]:
#apply model on validation set
df_val = pd.read_parquet('./Data/yellow_tripdata_2023-02.parquet')
df_val['duration'] = df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime
df_val.duration = df_val.duration.apply(lambda td: td.total_seconds() / 60)

df_val_clean = remove_outlier_prepare(df_val)




2913955
2855951
Fraction remaining: 0.9800944077722545
Percentage remaining: 98.00944077722545 %
(2855951, 514)
Number of columns: 514


In [65]:
y_val = df_val_clean['duration'].values
val_dicts = df_val_clean[categorical + numerical].to_dict(orient='records')
# dv = DictVectorizer()
X_val = dv.transform(val_dicts)
val_pred = lr.predict(X_val)
root_mean_squared_error(y_val, val_pred)

7.820173617322537